Testing 2 on present arms RL

Verify versions, cuda, etc

In [1]:
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [2]:
import torch
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import robosuite as suite
from robosuite import make
from robosuite.wrappers import GymWrapper

torch.__version__ , torch.cuda.is_available()

[robosuite WARNING] No private macro file found! (macros.py:57)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:58)
[robosuite WARNING] To setup, run: python /home/pranav/Documents/LearnFlake/src/external_pkgs/RoboSuite/robosuite/scripts/setup_macros.py (macros.py:59)


('2.5.1+cu124', False)

In [3]:
class RobosuiteGymWrapper(gym.Env):
    """
    Generic wrapper for using robosuite environment with Gymnasium for training reinforcement leanring (RL) policies
    """
    def __init__(self, env_name="Lift", robots="Jaco", has_renderer=False, use_camera_obs=False):
        super().__init__()
        self.env = GymWrapper(
            make(
                env_name=env_name, # Specify task or custom tasks
                robots=robots, # Robot(s) used
                has_renderer=has_renderer, # Toggles on-screen rendering
                has_offscreen_renderer=False if has_renderer else True, # Enables offscreen rendering for faster training (needs to be on if on-screen is off)
                use_camera_obs=use_camera_obs, # Toggles camera-based observations e.g. camera_names=["frontview"] TODO: wrapper with cameras
                reward_shaping=True, # e.g. partial rewards for sub-tasks
                # lite_physics=True
                # renderer="egl", # TODO: make compatible with OpenCV
            ) 
        ) # Converts robosuite env into a gym env

        self.action_space = self.env.action_space # TODO: custom robot action space
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=self.env.observation_space.shape, dtype=np.float32
        ) # TODO: add/preprocess observations ; sensors
    def reset(self, seed=None, options=None):
        """
        Reset the env with optional seeding.

        Args: 
            seed (int, optional): seed for reproducibility.
            options (dict, optional): additional options for reset (NotImplemented).
        
        Returns:
            obs (np.ndarray): initial observation
        """
        if seed is not None:
            self.env.seed(seed) # set seed in robosuite env

        return self.env.reset() # Method to reset env TODO: custom reset logic

    def step(self, action):
        result = self.env.step(action)  # Get the full output
        return result
    
    def render(self, mode="human"):
        return self.env.render() # TODO: custom rendering for specific cameras, offline rl
    
    def close(self):
        self.env.close() # Cleanup

In [4]:
# %pip install stable_baselines3 

In [5]:
# %pip install tqdm

In [6]:
from tqdm import tqdm
import time

# Training loop with progress
def train_with_progress(model, total_timesteps, log_interval=1000):
    steps=0
    with tqdm(total=total_timesteps, desc="Progress", unit="step") as pbar:
        # Train for a chunk fo steps
        while steps < total_timesteps:
            model.learn(total_timesteps=log_interval, reset_num_timesteps=False)
            steps += log_interval
            pbar.update(log_interval)
        # time.sleep(0.1)

Train with DDPG


In [7]:
from stable_baselines3 import DDPG
from stable_baselines3.common.noise import NormalActionNoise
import numpy as np
import os
import datetime

# Init env
env_name = "Lift"
robot_name = "Jaco"
env = RobosuiteGymWrapper(env_name=env_name, robots=robot_name, has_renderer=True)

# Add action noise for exploration
n_actions = env.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1*np.ones(n_actions))

# Init model
model = DDPG("MlpPolicy", env, action_noise=action_noise, verbose=1, learning_rate=1e-3)

# Train the model
train_with_progress(model, total_timesteps=10000) # Test trial

# Save model
date = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
save_dir = f"./models/{env_name}/{robot_name}/{date}"
os.makedirs(save_dir, exist_ok=True)

model.save(os.path.join(save_dir, "ddpg_pp"))
print(f"Model Saved to: {save_dir}/ddpg_pp.zip")


[robosuite INFO] Loading controller configuration from: /home/pranav/Documents/LearnFlake/src/external_pkgs/RoboSuite/robosuite/controllers/config/default/composite/basic.json (composite_controller_factory.py:121)
[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the c

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


Progress:   0%|          | 0/10000 [00:00<?, ?step/s][robosuite INFO] Loading controller configuration from: /home/pranav/Documents/LearnFlake/src/external_pkgs/RoboSuite/robosuite/controllers/config/default/composite/basic.json (composite_controller_factory.py:121)
[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)

KeyboardInterrupt: 

In [ ]:
from robosuite import make

env = make(
    env_name="PickPlace",
    robots="Jaco",
    has_renderer=True,  # Enable visible rendering
    has_offscreen_renderer=False,  # Disable offscreen rendering
    use_camera_obs=False,  # Disable camera observations
    reward_shaping=True,
    lite_physics=True
)
env.reset()
env.render()


[robosuite INFO] Loading controller configuration from: C:\Users\camer\robosuite\robosuite\controllers\config\default\composite\basic.json (composite_controller_factory.py:121)


[robosuite WARNING] The config has defined for the controller "left", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for left from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "torso", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for torso from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "head", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for head from self.part_controller_config. (robot.py:151)
[robosuite WARNING] The config has defined for the controller "base", but the robot does not have this component. Skipping, but make sure this is intended.Removing the controller config for base from self.part_controller_config. (robot.py:151)
[robosuite WARNING] Th

In [ ]:
import mujoco
mujoco.__version__

'3.2.6'